In [1]:
#Importing the necessary libraries 
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import wordnet
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.tokenize import word_tokenize
import numpy as np
import pandas as pd
import re
import sys
import unicodedata


#Importing the data into a dataframe
comments = pd.read_csv('DailyComments.csv')
comments

,Day of Week,comments
0,Monday,"Hello, how are you?"
1,Tuesday,Today is a good day!
2,Wednesday,It's my birthday so it's a really special day!
3,Thursday,Today is neither a good day or a bad day!
4,Friday,I'm having a bad day.
5,Saturday,There' s nothing special happening today.
6,Sunday,Today is a SUPER good day!


In [2]:
#Before a scheme for sentiment analysis can be applied, the text must be cleaned

#Converting all text to lowercase letters
comments['comments'] = comments['comments'].str.lower()
comments['comments']

0                               hello, how are you?
1                              today is a good day!
2    it's my birthday so it's a really special day!
3         today is neither a good day or a bad day!
4                             i'm having a bad day.
5         there' s nothing special happening today.
6                        today is a super good day!
Name: comments, dtype: object

In [3]:
#Removing all punctuation from the text

punctuation = dict.fromkeys(i for i in range(sys.maxunicode) if unicodedata.category(chr(i)).startswith('P'))

comments['comments'] = [string.translate(punctuation) for string in comments['comments']]
comments['comments']

0                              hello how are you
1                            today is a good day
2    its my birthday so its a really special day
3       today is neither a good day or a bad day
4                            im having a bad day
5        there s nothing special happening today
6                      today is a super good day
Name: comments, dtype: object

In [4]:
#Tokenizing the words and appending each list of tokenized words into a list to be added to the dataframe
tokenized_list = []
for string in comments['comments']:
    tokenized = word_tokenize(string)
    tokenized_list.append(tokenized)

comments['comments'] = tokenized_list
comments['comments']

0                               [hello, how, are, you]
1                            [today, is, a, good, day]
2    [its, my, birthday, so, its, a, really, specia...
3    [today, is, neither, a, good, day, or, a, bad,...
4                            [im, having, a, bad, day]
5       [there, s, nothing, special, happening, today]
6                     [today, is, a, super, good, day]
Name: comments, dtype: object

In [5]:
#Applying POS tagging to each comment
comments_tagged = [pos_tag(comment) for comment in comments['comments']]
comments_tagged


[[('hello', 'VB'), ('how', 'WRB'), ('are', 'VBP'), ('you', 'PRP')],
 [('today', 'NN'), ('is', 'VBZ'), ('a', 'DT'), ('good', 'JJ'), ('day', 'NN')],
 [('its', 'PRP$'),
  ('my', 'PRP$'),
  ('birthday', 'NN'),
  ('so', 'IN'),
  ('its', 'PRP$'),
  ('a', 'DT'),
  ('really', 'RB'),
  ('special', 'JJ'),
  ('day', 'NN')],
 [('today', 'NN'),
  ('is', 'VBZ'),
  ('neither', 'CC'),
  ('a', 'DT'),
  ('good', 'JJ'),
  ('day', 'NN'),
  ('or', 'CC'),
  ('a', 'DT'),
  ('bad', 'JJ'),
  ('day', 'NN')],
 [('im', 'NN'), ('having', 'VBG'), ('a', 'DT'), ('bad', 'JJ'), ('day', 'NN')],
 [('there', 'EX'),
  ('s', 'VBZ'),
  ('nothing', 'NN'),
  ('special', 'JJ'),
  ('happening', 'VBG'),
  ('today', 'NN')],
 [('today', 'NN'),
  ('is', 'VBZ'),
  ('a', 'DT'),
  ('super', 'JJ'),
  ('good', 'JJ'),
  ('day', 'NN')]]

In [6]:
#Applying text lemmatization using a series of functions

#Converting tags from a POS tag vector into letters to run through the lemmatizer function
def pos_tagger(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:          
        return None

#Function that takes a list of POS tag vectors and switches the POS tags for 
#letters that can be input into the lemmatizer function
def POS_simplifier(pos_tag_list):
    lemmatizer = WordNetLemmatizer()
    wordnet_tagged = [list(map(lambda x: (x[0], pos_tagger(x[1])), comments_tagged[i])) for i in range(len(pos_tag_list))]            
    return wordnet_tagged
                    

In [7]:
comments_tagged_simple = POS_simplifier(comments_tagged)
comments_tagged_simple

[[('hello', 'v'), ('how', None), ('are', 'v'), ('you', None)],
 [('today', 'n'), ('is', 'v'), ('a', None), ('good', 'a'), ('day', 'n')],
 [('its', None),
  ('my', None),
  ('birthday', 'n'),
  ('so', None),
  ('its', None),
  ('a', None),
  ('really', 'r'),
  ('special', 'a'),
  ('day', 'n')],
 [('today', 'n'),
  ('is', 'v'),
  ('neither', None),
  ('a', None),
  ('good', 'a'),
  ('day', 'n'),
  ('or', None),
  ('a', None),
  ('bad', 'a'),
  ('day', 'n')],
 [('im', 'n'), ('having', 'v'), ('a', None), ('bad', 'a'), ('day', 'n')],
 [('there', None),
  ('s', 'v'),
  ('nothing', 'n'),
  ('special', 'a'),
  ('happening', 'v'),
  ('today', 'n')],
 [('today', 'n'),
  ('is', 'v'),
  ('a', None),
  ('super', 'a'),
  ('good', 'a'),
  ('day', 'n')]]

In [8]:
lemmatizer = WordNetLemmatizer()
lemmatized_sentences = []
for sublist in comments_tagged_simple:
    lemmatized_sentences_sub = []
    for word, tag in sublist:
        if tag is None:
            # if there is no available tag, append the token as is
            lemmatized_sentences_sub.append(word)
        else:        
            # else use the tag to lemmatize the token
            lemmatized_sentences_sub.append(lemmatizer.lemmatize(word, tag))
    lemmatized_sentences.append(lemmatized_sentences_sub)

In [9]:
lemmatized_sentences

[['hello', 'how', 'be', 'you'],
 ['today', 'be', 'a', 'good', 'day'],
 ['its', 'my', 'birthday', 'so', 'its', 'a', 'really', 'special', 'day'],
 ['today', 'be', 'neither', 'a', 'good', 'day', 'or', 'a', 'bad', 'day'],
 ['im', 'have', 'a', 'bad', 'day'],
 ['there', 's', 'nothing', 'special', 'happen', 'today'],
 ['today', 'be', 'a', 'super', 'good', 'day']]

In [12]:
def listToString(s): 
    str1 = " "   
    return str1.join(s)

In [13]:
new_list = []
for sublist in lemmatized_sentences:
    new_list.append(listToString(sublist))

In [14]:
new_list

['hello how be you',
 'today be a good day',
 'its my birthday so its a really special day',
 'today be neither a good day or a bad day',
 'im have a bad day',
 'there s nothing special happen today',
 'today be a super good day']

In [15]:
from textblob import TextBlob

for comment in new_list:
    analysis = TextBlob(comment)
    print(analysis.sentiment)

Sentiment(polarity=0.0, subjectivity=0.0)
Sentiment(polarity=0.7, subjectivity=0.6000000000000001)
Sentiment(polarity=0.35714285714285715, subjectivity=0.5714285714285714)
Sentiment(polarity=5.551115123125783e-17, subjectivity=0.6333333333333333)
Sentiment(polarity=-0.6999999999999998, subjectivity=0.6666666666666666)
Sentiment(polarity=0.35714285714285715, subjectivity=0.5714285714285714)
Sentiment(polarity=0.5166666666666666, subjectivity=0.6333333333333333)


In [16]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()
for comment in new_list:
    print(analyzer.polarity_scores(comment))

{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
{'neg': 0.0, 'neu': 0.58, 'pos': 0.42, 'compound': 0.4404}
{'neg': 0.0, 'neu': 0.728, 'pos': 0.272, 'compound': 0.4576}
{'neg': 0.425, 'neu': 0.575, 'pos': 0.0, 'compound': -0.7101}
{'neg': 0.467, 'neu': 0.533, 'pos': 0.0, 'compound': -0.5423}
{'neg': 0.311, 'neu': 0.689, 'pos': 0.0, 'compound': -0.3089}
{'neg': 0.0, 'neu': 0.37, 'pos': 0.63, 'compound': 0.7783}
